In [1]:
%pip install tatc
from tatc import utils
from tatc.schemas import PointedInstrument, Satellite, Instrument, TwoLineElements  

earthcare = Satellite(
    name = "EarthCare",
    orbit = TwoLineElements(
        tle=[
            "1 59908U 24101A   25200.34125573  .00010433  00000+0  14571-3 0  9999",
            "2 59908  97.0168 326.4971 0001222 108.6708 251.4681 15.57041891 64775"
        ]
    ),
    instruments=[
        #PointedInstrument(
         #   name = "CPR",
            #field_of_regard=utils.swath_width_to_field_of_regard(394e3,650),
          #  cross_track_field_of_view = utils.swath_width_to_field_of_regard(394e3, 6500),
           # along_track_field_of_view = utils.swath_width_to_field_of_regard(394e3,10000)
        #),
        PointedInstrument(
            name = "MSI",
            field_of_regard=utils.swath_width_to_field_of_regard(394e3,150e3) + 2*5.760868, #cross_track_field_of_view + 2*roll angle
            cross_track_field_of_view = utils.swath_width_to_field_of_view(394e3, 150e3, 5.760868),
            along_track_field_of_view = utils.swath_width_to_field_of_view(394e3, 10e3, 5.760868),
            roll_angle = 5.760868,  # degrees
            is_rectangular = True  
        )
    ]
)
        
satellites = [earthcare]


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datetime import datetime, timezone, timedelta
from dateutil.relativedelta import relativedelta
from tatc.analysis import compute_ground_track
import pandas as pd
from joblib import Parallel, delayed

startdate = datetime(2025,7,19,15,4, tzinfo = timezone.utc)  # initial date, discard the year. 
# span the full G5NR Nature Run (~2 years). generous upper bound here;
# cell 5 trims duration + ground_tracks to the dataset's exact hourly axis
# so every frame maps onto a real cloud field.
duration = timedelta(days = 763)
step = timedelta(seconds = 5)
batch_duration = timedelta(minutes=10)

def compute_groundtrack(satellite, start, duration, batch_duration, time_step):
    return pd.concat(
    Parallel(-1)(
        delayed(compute_ground_track)(
            satellite,
            pd.date_range(start + i*batch_duration, start + (i+1)*batch_duration, freq=time_step, inclusive="left"),
            crs="spice"
        ) 
        for i in range( duration // batch_duration)
        for satellite in satellites
    ),
    ignore_index=True
)

ground_tracks = compute_groundtrack(satellites[0], startdate, duration, batch_duration, step)


In [ ]:
print(ground_tracks.head())
print(ground_tracks.columns)
print(len(ground_tracks))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from cartopy import crs as ccrs
from IPython.display import HTML
from matplotlib.patches import Patch

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = batch_duration

# a full 2-year run has ~17k hourly / ~105k 10-min frames. an inline jshtml
# animation that large bloats the notebook to GBs and hangs the browser, so
# cap the preview to the first ANIM_MAX_FRAMES frames. raise/remove to render more.
ANIM_MAX_FRAMES = 300

def animate(frame):
    ax.clear()
    time = startdate + frame*frame_duration
    tracks = ground_tracks[
        (time <= ground_tracks.time) 
        & (ground_tracks.time < time + frame_duration)
    ]
    if not tracks.empty:
        tracks.plot(ax=ax, color="r", transform=ccrs.PlateCarree())

    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    ax.set_title(time)
    fig.tight_layout()

ani = animation.FuncAnimation(
    fig,
    animate, 
    frames=min(duration // frame_duration, ANIM_MAX_FRAMES), 
    interval=100, 
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
grid_size = 0.5
g5nr_frame_duration = timedelta(hours=1)

In [ ]:
import rioxarray
import xarray as xr

dataset = xr.open_dataset(
    "https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.5000_deg/tavg/tavg01hr_2d_met3_Cx",
    decode_times=True,
)
#xr.open_dataset('https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.0625_deg/tavg/tavg30mn_2d_met3_Nx').to_netcdf('dataset.nc')
#dataset = xr.open_dataset('dataset.nc', decode_times=True)
dataset.rio.write_crs("epsg:4326", inplace=True)
dataset.rio.set_spatial_dims("lon", "lat", inplace=True)

import pandas as pd
# real g5nr hourly axis (tz-naive). the shared `frame` index maps directly
# onto this axis: g5nr_time(frame) = g5nr_start + frame * g5nr_frame_duration,
# so cloud data walks the true 2005-2007 span instead of one collapsed day.
g5nr_times = pd.to_datetime(dataset["time"].values)
g5nr_start = g5nr_times[0].to_pydatetime()
n_g5nr = len(g5nr_times)

# trim duration + ground_tracks to the exact data window so every hourly frame
# has real cloud data (avoids phantom non-observing passes past the axis end).
duration = n_g5nr * g5nr_frame_duration
ground_tracks = ground_tracks[
    ground_tracks.time < startdate + duration
].reset_index(drop=True)
print(f"g5nr axis: {n_g5nr} hourly slices, {g5nr_times[0]} -> {g5nr_times[-1]}")
print(f"trimmed ground_tracks to {len(ground_tracks)} rows over {duration}")


In [ ]:
# decorate ground_tracks with tautot looked up from g5nr at each row's
# centroid + time. mirrors the time-mapping trick used in get_clusters so
# the lookup hits the same g5nr time index the clusters do.
def lookup_tautot(ground_tracks, dataset, startdate, g5nr_frame_duration):
    frames = ((ground_tracks.time - startdate) // g5nr_frame_duration).astype(int)
    ds_times = pd.to_datetime([
        g5nr_start + min(int(f), n_g5nr - 1) * g5nr_frame_duration
        for f in frames
    ])
    times_da = xr.DataArray(ds_times.values, dims="points")
    lats_da  = xr.DataArray(ground_tracks.geometry.centroid.y.values, dims="points")
    lons_da  = xr.DataArray(ground_tracks.geometry.centroid.x.values, dims="points")
    return dataset["tautot"].sel(
        time=times_da, lat=lats_da, lon=lons_da, method="nearest"
    ).values

ground_tracks["tautot"] = lookup_tautot(
    ground_tracks, dataset, startdate, g5nr_frame_duration
)
display(ground_tracks.head())

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box
import scipy.ndimage as ndi

def compute_grid_cell_area(lat, lon):
    R = 6371.0  # Earth radius in km
    d2r = np.pi / 180.0
    ny, nx = len(lat), len(lon)

    dlat = abs(lat[1] - lat[0]) if len(lat) > 1 else 1.0
    dlon = abs(lon[1] - lon[0]) if len(lon) > 1 else 1.0

    lat_rad = lat * d2r
    dlat_rad = dlat * d2r
    dlon_rad = dlon * d2r

    area = np.zeros((ny, nx), dtype=np.float64)
    for i in range(ny):
        area[i, :] = (
            (R**2) * dlon_rad * 
            (np.sin(lat_rad[i] + dlat_rad / 2) - np.sin(lat_rad[i] - dlat_rad / 2))
        )
    return area


In [ ]:
import scipy.ndimage as ndi
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import time as _time
import gc
from shapely.geometry import box
from joblib import Parallel, delayed

# parallelized get_clusters: frame -> real g5nr-time, batched OpenDap
# fetch per chunk, then per-frame CPU work in threads.

threshold = 220  # brightness temperature threshold in Kelvin

# tunables
IO_CHUNK_TIMES = 168   # distinct g5nr hours per OpenDap request
IO_RETRIES     = 3     # exponential backoff on rate-limit / transient errors
CPU_WORKERS    = -1    # threads for per-frame work (numpy/scipy release GIL)


def _frame_ds_time(frame):
    # walk the real g5nr hourly axis (no single-day collapse); clamp any
    # overshoot to the final slice.
    return g5nr_start + min(int(frame), n_g5nr - 1) * g5nr_frame_duration


def _fetch_chunk(chunk_times):
    # one batched HTTPs request via a contiguous time slice, then local pick
    t_min, t_max = min(chunk_times), max(chunk_times)
    for attempt in range(IO_RETRIES):
        try:
            sub = (
                dataset[["lwtup", "prectot"]]
                .sel(time=slice(t_min, t_max), lat=slice(-89, 89))
                .load()
            )
            break
        except Exception:
            if attempt == IO_RETRIES - 1:
                raise
            _time.sleep(2 ** attempt)
    pick = xr.DataArray(pd.to_datetime(chunk_times).values, dims="t")
    sub = sub.sel(time=pick, method="nearest")
    return {
        t: (sub["lwtup"].isel(t=j).values, sub["prectot"].isel(t=j).values)
        for j, t in enumerate(chunk_times)
    }


def _process_frame(frame, arrays, area_2d, lat2d, lon2d):
    # CPU-only: build the dissolved cluster GeoDataFrame for one frame
    try:
        lwtup_2d, prectot_2d = arrays[_frame_ds_time(frame)]

        tb = np.sqrt(np.sqrt(lwtup_2d / 5.67037e-8))
        labels, _ = ndi.label(tb < threshold)

        mask = labels > 0
        if not mask.any():
            return gpd.GeoDataFrame()

        cluster_labels = labels[mask]
        lat_vals       = lat2d[mask]
        lon_vals       = lon2d[mask]
        prectot_vals   = prectot_2d[mask]
        area_vals      = area_2d[mask]

        # single NaN filter across all columns — keeps lengths aligned
        finite = ~np.isnan(prectot_vals)
        cluster_labels = cluster_labels[finite]
        lat_vals       = lat_vals[finite]
        lon_vals       = lon_vals[finite]
        prectot_vals   = prectot_vals[finite]
        area_vals      = area_vals[finite]

        if cluster_labels.size == 0:
            return gpd.GeoDataFrame()

        cells = gpd.GeoDataFrame(
            {
                "count": 1,
                "cluster": cluster_labels,
                "lat": lat_vals,
                "lon": lon_vals,
                "time": startdate + frame * g5nr_frame_duration,
                "prectot": prectot_vals,
                "area": area_vals,
            },
            geometry=[
                box(lo, la, lo + grid_size, la + grid_size)
                for lo, la in zip(lon_vals, lat_vals)
            ],
            crs="EPSG:4326",
        )
        cells["tot_prectot"] = cells["prectot"]
        cells["avg_prectot"] = cells["prectot"]
        cells["max_prectot"] = cells["prectot"]

        return cells[cells.cluster > 0].dissolve(
            by=["time", "cluster"],
            aggfunc={
                "count": "sum",
                "area": "sum",
                "tot_prectot": "sum",
                "avg_prectot": "mean",
                "max_prectot": "max",
            },
        )
    except Exception as e:
        print(f"Frame {frame} failed: {e}")
        return gpd.GeoDataFrame()


_t0 = _time.time()
# duration is already trimmed to n_g5nr hours (cell 5); clamp is belt-and-suspenders.
n_frames = min(int(duration // g5nr_frame_duration), n_g5nr)

# precompute area + meshgrid once, shared across every frame
sample_lat = dataset["lat"].sel(lat=slice(-89, 89)).values
sample_lon = dataset["lon"].values
area_2d = compute_grid_cell_area(sample_lat, sample_lon)
lon2d, lat2d = np.meshgrid(sample_lon, sample_lat)

# with the real-axis mapping each frame maps ~1:1 to a distinct g5nr hour;
# the set() still guards against any duplicates from the overshoot clamp.
unique_times = sorted({_frame_ds_time(f) for f in range(n_frames)})
print(f"{n_frames} frames -> {len(unique_times)} distinct g5nr time slices")

# group frames by which time-chunk their data lives in (bounds memory)
time_chunks = [
    unique_times[i:i + IO_CHUNK_TIMES]
    for i in range(0, len(unique_times), IO_CHUNK_TIMES)
]
time_to_chunk_idx = {t: ci for ci, c in enumerate(time_chunks) for t in c}
frames_by_chunk = [[] for _ in time_chunks]
for f in range(n_frames):
    frames_by_chunk[time_to_chunk_idx[_frame_ds_time(f)]].append(f)

# stream: fetch chunk -> process its frames in parallel -> drop arrays
all_results = []
for ci, chunk_times in enumerate(time_chunks):
    _tc = _time.time()
    arrays = _fetch_chunk(chunk_times)
    chunk_results = Parallel(n_jobs=CPU_WORKERS, backend="threading")(
        delayed(_process_frame)(f, arrays, area_2d, lat2d, lon2d)
        for f in frames_by_chunk[ci]
    )
    all_results.extend(chunk_results)
    del arrays, chunk_results
    gc.collect()
    print(f"chunk {ci+1}/{len(time_chunks)}: "
          f"{len(chunk_times)} times, {len(frames_by_chunk[ci])} frames, "
          f"{_time.time()-_tc:.1f}s")

clusters = pd.concat([r for r in all_results if not r.empty]).reset_index()
print(f"total: {_time.time()-_t0:.1f}s, {len(clusters)} cluster rows")
display(clusters)

In [ ]:
from skyfield.api import wgs84
from tatc.constants import de421, timescale

# compute solar hour based on the angle of the sun as seen at the feature centroid
clusters["solar_hour"] = clusters.apply(
    lambda r: (
        de421["earth"] + wgs84.latlon(r.geometry.centroid.y, r.geometry.centroid.x)
    )
    .at(timescale.from_datetime(r.time + frame_duration/2))
    .observe(de421["sun"])
    .apparent()
    .hadec()[0]
    .hours
    + 12,
    axis=1,
)
display(clusters)

In [ ]:
import shapely

# bin times to the g5nr hour grid that both clusters and ground_tracks live on.
# this is the same (time - startdate) // g5nr_frame_duration trick used in
# lookup_tautot (above) and in get_clusters, so the bin a cluster row falls
# in is exactly the bin its 1-hour observation window covers — the per-bin
# dissolve below is semantically equivalent to the original per-row apply,
# but pays the unary_union cost ~5,880 times instead of ~1.45M times.
clusters_hbin = ((clusters.time - startdate) // g5nr_frame_duration).astype(int)
gt_hbin       = ((ground_tracks.time - startdate) // g5nr_frame_duration).astype(int)

# dissolve ground_tracks once per hour bin (one union geometry per hour)
gt_per_hbin = ground_tracks.assign(hbin=gt_hbin).dissolve(by="hbin")[["geometry"]]

# left-merge the per-bin union onto each cluster row.
# suffixes=("", "_gt") keeps clusters' own "geometry" column intact and
# puts the ground-track union under "geometry_gt".
merged = clusters.assign(hbin=clusters_hbin).merge(
    gt_per_hbin, left_on="hbin", right_index=True, how="left",
    suffixes=("", "_gt"),
)

# vectorized intersects via shapely 2.x.
# rows whose hbin has no ground-track passes get a None on the right side and stay 0.
left_geom  = merged["geometry"].values
right_geom = merged["geometry_gt"].values
has_gt   = np.array([g is not None for g in right_geom])
observed = np.zeros(len(clusters), dtype=int)
observed[has_gt] = shapely.intersects(left_geom[has_gt], right_geom[has_gt]).astype(int)
clusters["observed"] = observed

display(clusters)
print(clusters[clusters["observed"] == 1], "features observed")


In [ ]:
import matplotlib.animation as animation
from cartopy import crs as ccrs
from IPython.display import HTML

# create a figure
fig, ax = plt.subplots(figsize=(8,4), subplot_kw={"projection": ccrs.PlateCarree()})

def animate(frame):
    ax.clear()
    time = startdate + frame*g5nr_frame_duration
    active_clusters = clusters[
        (clusters.time >= time) 
        & (clusters.time < time + g5nr_frame_duration)
    ]
    if not active_clusters.empty:
        active_clusters.plot(
            ax=ax, 
            column="observed", 
            vmin=0, 
            vmax=1, 
            cmap="RdYlGn"
        )
    track = ground_tracks[
        (ground_tracks.time >= time) 
        & (ground_tracks.time < time + g5nr_frame_duration)
    ]
    if not track.empty:
        track.dissolve().plot(ax=ax, color="black", alpha = 0.2)
    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    fig.tight_layout()
    plt.legend(handles=[
        Patch(label="MSI", facecolor="black", alpha=0.8),
        Patch(label="Unobserved", facecolor="#A50026"),
        Patch(label="Observed", facecolor="#006837"),
    ], loc="lower right")

ani = animation.FuncAnimation(
    fig, 
    animate, 
    frames=min(duration//g5nr_frame_duration, ANIM_MAX_FRAMES), 
    interval=100, 
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
# define a frame number for both datasets — used by the observed-only
# sjoin in the next cell via on_attribute="frame" for hour-binned matching
ground_tracks["frame"] = (ground_tracks.time - startdate) // g5nr_frame_duration
clusters["frame"]      = (clusters.time      - startdate) // g5nr_frame_duration


In [ ]:
# observed + non-observing join: each row is a 10-minute satellite pass.
# passes that scooped >=1 observed cluster (regardless of that cluster's
# prectot value) carry cluster-derived stats; the rest are negative
# training samples with zero cluster stats but valid pass-level features
# (time / solar_hour / lat / lon / satellite / tautot) so the RL pipeline
# can learn to not-prioritize clear-sky passes.
from skyfield.api import wgs84
from tatc.constants import de421, timescale

def _solar_hour_at(lat, lon, dt):
    # same computation cell 9 does for cluster.solar_hour, applied to
    # arbitrary (lat, lon, dt) — used to backfill non-observing rows.
    # skyfield requires tz-aware datetimes; the derived _agg["time"] can
    # come back tz-naive from pandas arithmetic, but it is UTC by
    # construction (startdate is UTC), so localize if needed.
    dt = pd.Timestamp(dt)
    if dt.tzinfo is None:
        dt = dt.tz_localize("UTC")
    dt = dt.to_pydatetime()
    return (
        (de421["earth"] + wgs84.latlon(lat, lon))
        .at(timescale.from_datetime(dt + frame_duration / 2))
        .observe(de421["sun"])
        .apparent()
        .hadec()[0]
        .hours
    ) + 12

observed_clusters = clusters[clusters["observed"] == 1].copy()

# cluster-centroid lat/lon — the agg below produces mean lat/lon per pass
# for observed rows. non-observing rows get NaN here and are backfilled
# with the ground_track's own centroid later. geographic CRS emits a
# benign UserWarning; error is well below the 0.5° grid cell.
observed_clusters["lat"] = observed_clusters.geometry.centroid.y
observed_clusters["lon"] = observed_clusters.geometry.centroid.x

joined_observed = gpd.sjoin(
    observed_clusters, ground_tracks,
    how="right",
    predicate="intersects",
    on_attribute="frame",
)

# Cast to a plain pandas DataFrame BEFORE the agg. The agg's internal
# concat({col: result, ...}, axis=1, keys=keys_to_use) builds MultiIndex
# columns temporarily, then calls __finalize__ on the result. When the
# input was a GeoDataFrame, geopandas's __finalize__ runs the safety
# check `(self.columns == self._geometry_column_name).sum() > 1` — and
# comparing a MultiIndex of tuples to the scalar "geometry" raises
# 'ValueError: truth value of an array with more than one element is
# ambiguous'. Passing a plain DataFrame skips that check; re-wrap as
# GeoDataFrame after.
joined_df = pd.DataFrame(joined_observed)

# key trick: derive `time` from `frame` instead of using `time_left`.
# on_attribute="frame" makes `frame` a merge KEY (like pandas merge on=),
# so it appears once, unsuffixed, and is populated for every row: the
# shared value for matched rows, the ground_track's own frame for
# unmatched (non-observing) rows. deriving time from it therefore keeps
# observed rows' previous time_left values byte-for-byte and gives
# non-observing rows a well-defined g5nr-aligned time.
#
# tautot bug fix: use "first" (per-pass scalar). sjoin duplicates the
# ground_track row once per intersecting cluster, so the old
# tot_tautot=sum inflated by n_clusters. tot/avg/max collapse into one
# `tautot` column since all three were the same underlying value.
_agg = joined_df.groupby(level=0).agg(
    geometry    =("geometry",    "first"),
    frame       =("frame",       "first"),
    satellite   =("satellite",   "first"),
    tautot      =("tautot",      "first"),
    lat         =("lat",         "mean"),
    lon         =("lon",         "mean"),
    solar_hour  =("solar_hour",  "mean"),
    count       =("count",       "sum"),
    area        =("area",        "sum"),
    tot_prectot =("tot_prectot", "sum"),
    avg_prectot =("avg_prectot", "mean"),
    max_prectot =("max_prectot", "max"),
)

# the sjoin's `frame` column comes from the LEFT (clusters) side, so it
# is NaN for non-observing rows — which made the derived time NaT and
# crashed the skyfield backfill. ground_tracks["frame"] shares _agg's
# index (how="right" keeps the right index) and is always populated:
# use it as the source of truth. for observed rows the two agree by the
# on_attribute="frame" equality constraint, so observed values are
# unchanged.
_agg["frame"] = ground_tracks["frame"]

# derive time uniformly from frame (g5nr-aligned for all rows).
# utc=True guarantees a tz-aware column regardless of what dtype the
# frame arithmetic produced — downstream cells compare against tz-aware
# timestamps (startdate + k*frame_duration) and would raise on naive.
_agg["time"] = pd.to_datetime(startdate + _agg["frame"] * g5nr_frame_duration, utc=True)

# ground-track fallback for lat/lon/solar_hour on non-observing rows.
# _agg is indexed by ground_track index (from how="right"), and
# ground_tracks.geometry.centroid.{x,y} share that index, so .fillna
# aligns by index automatically.
gt_cent_y = ground_tracks.geometry.centroid.y
gt_cent_x = ground_tracks.geometry.centroid.x
_agg["lat"] = _agg["lat"].fillna(gt_cent_y)
_agg["lon"] = _agg["lon"].fillna(gt_cent_x)

missing = _agg["solar_hour"].isna()
if missing.any():
    _agg.loc[missing, "solar_hour"] = [
        _solar_hour_at(_agg.at[i, "lat"], _agg.at[i, "lon"], _agg.at[i, "time"])
        for i in _agg.loc[missing].index
    ]

# cluster stats: 0 for non-observing passes.
for c in ["count", "area", "tot_prectot", "avg_prectot", "max_prectot"]:
    _agg[c] = _agg[c].fillna(0)

# explicit observed indicator. count > 0 iff any observed cluster was
# intersected (including observed-but-zero-prectot clusters, which get
# a count>0 contribution from their pixels).
_agg["observed"] = (_agg["count"] > 0).astype(int)

aggregated_observed = gpd.GeoDataFrame(
    _agg, geometry="geometry", crs=ground_tracks.crs,
)

display(aggregated_observed)
print(len(aggregated_observed), "total passes;",
      int(aggregated_observed.observed.sum()),
      "observed (intersected >=1 observed cluster)")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
from cartopy import crs as ccrs
from IPython.display import HTML

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = batch_duration

# LogNorm can't span 0 — use smallest positive value as vmin
_positive = aggregated_observed.loc[aggregated_observed['avg_prectot'] > 0, 'avg_prectot']
prectot_vmin = _positive.min()
prectot_vmax = _positive.max()

# first plot to generate a colorbar
aggregated_observed.plot(
    ax=ax, 
    column="avg_prectot", 
    norm=mcolors.LogNorm(prectot_vmin, prectot_vmax),
    transform=ccrs.PlateCarree(), 
    legend=True, 
    legend_kwds={"orientation": "horizontal", "label": "Average Precipitation (kg/m$^2$/s)"}
)

def animate(frame):
    ax.clear()
    time = startdate + frame*frame_duration
    tracks = aggregated_observed[
        (aggregated_observed.time >= time) 
        & (aggregated_observed.time < time + frame_duration)
    ]
    if not tracks.empty:
        tracks.boundary.plot(ax=ax, color="r", linewidth=0.5)
        tracks.plot(
            ax=ax, 
            column="avg_prectot", 
            norm=mcolors.LogNorm(prectot_vmin, prectot_vmax),
            transform=ccrs.PlateCarree(),
        )
    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    ax.set_title(time)
    fig.tight_layout()

ani = animation.FuncAnimation(
    fig,
    animate, 
    frames=min(duration // frame_duration, ANIM_MAX_FRAMES), 
    interval=100, 
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

clusters_filtered = clusters[
    (clusters['time'].dt.day == 19)
]
hourly_stats = (
    clusters_filtered[:1100]
    .groupby('solar_hour')
    .agg(mean_prectot=('avg_prectot', 'mean'))
    .reset_index()
    .sort_values('solar_hour')
)

print(hourly_stats.head())
print(len(hourly_stats))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly_stats['solar_hour'], hourly_stats['mean_prectot'], marker = '.', color = 'b')
ax.set_xlabel('Solar Hour (0–23)')
ax.set_ylabel('AVG_PRECTOT (kg m⁻² s⁻¹)')
ax.set_title('Mean Precipitation vs Solar Hour for 1 month period')
ax.set_xticks(range(0, 25, 2))
plt.tight_layout()
plt.show()

#probability distribution (KDE)


In [ ]:
%pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

sns.kdeplot(
    data=clusters_filtered[:1100],   
    x="solar_hour",
    y="avg_prectot",
    fill=True,      
    cmap="Reds",  
    bw_adjust=0.5,       
    ax=ax,
)

ax.set_xlabel("Solar Hour (0–24)")
ax.set_ylabel("AVG_PRECTOT (kg m⁻² s⁻¹)")
ax.set_title("Mean Precipitation vs Solar Hour for 1 month period")
ax.set_xlim(0, 23)
ax.set_xticks(range(0, 25, 2))
ax.set_ylim(0, 4e-3)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

agg = aggregated_observed.copy()
agg["centroid"] = agg.geometry.centroid 

flat = pd.DataFrame({
    "lon": agg.centroid.x,
    "lat": agg.centroid.y,
    "avg_prectot": agg.avg_prectot,
    "tot_prectot": agg.tot_prectot,
    "time": agg.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="avg_prectot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)



ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of  Simulated Mean Precipitation observable by EarthCare MSI for 2 years")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
#plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

agg = aggregated_observed.copy()
agg["centroid"] = agg.geometry.centroid 

flat = pd.DataFrame({
    "lon": agg.centroid.x,
    "lat": agg.centroid.y,
    "tautot": agg.tautot,
    "time": agg.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="tautot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)


ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of Average Cloud Cover Measured by EarthCare \n MSI in observed clusters within satellite swath for 2 years")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
#plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

gd_tau = ground_tracks.copy()
gd_tau["centroid"] = gd_tau.geometry.centroid 

flat = pd.DataFrame({
    "lon": gd_tau.centroid.x,
    "lat": gd_tau.centroid.y,
    "tautot": gd_tau.tautot,
    "time": gd_tau.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="tautot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)


ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of Average Cloud Cover Measured by EarthCare MSI for 2 years")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
#plt.tight_layout()
plt.show()

In [ ]:
print(f"all ground-track passes: {len(ground_tracks):>6}")
print(f"observed-only passes:    {int((aggregated_observed.observed == 1).sum()):>6}")
print(f"observed fraction:       {(aggregated_observed.observed == 1).sum() / len(ground_tracks):.1%}")

In [ ]:
per_frame = clusters[clusters["observed"]==1].groupby(
    ((clusters["time"] - startdate) // g5nr_frame_duration).astype(int)
).size()
print(per_frame.describe())

In [ ]:
obs = clusters[clusters["observed"]==1]
print("centroid lat distribution of observed clusters:")
print(obs.geometry.centroid.y.describe())

In [ ]:
# sanity check 1: are observed clusters spread reasonably across frames?
# expectation: ~total_observed / n_frames per frame on average, with a
# reasonable spread. a giant pile-up at zero or a few frames hoarding the
# bulk would point to a binning bug in the vectorized observed rewrite.
obs = clusters[clusters["observed"] == 1]
hbin = ((obs["time"] - startdate) // g5nr_frame_duration).astype(int)
per_frame = obs.groupby(hbin).size()
print("observed clusters per g5nr frame:")
print(per_frame.describe())
print(f"\nframes containing at least one observed cluster: {len(per_frame)}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(per_frame.values, bins=40)
ax.set_xlabel("observed clusters per frame")
ax.set_ylabel("count of frames")
ax.set_title("Per-frame distribution of observed clusters")
plt.tight_layout()
plt.show()


In [ ]:
# sanity check 2: latitude distribution of observed clusters.
# expectation for real deep convection: peak in the tropics (|lat| < 30°)
# with a secondary mode in mid-latitude storm tracks.
# EarthCARE's sun-sync orbit at ~97 deg inclination geometrically over-samples
# latitudes near +/- 82 deg (where ground tracks converge), but cold-cloud
# clusters concentrate in the tropics — so the *weighted* observed
# distribution should still skew tropical. heavy concentration above |lat|=60
# usually means cold polar surfaces (Antarctic ice, sea-ice edges) are being
# detected by the tb<220K threshold as if they were deep convective cloud tops.
obs = clusters[clusters["observed"] == 1]
lats = obs.geometry.centroid.y
print("centroid lat distribution of observed clusters:")
print(lats.describe())

bands = [
    ("south polar  (lat <= -75)",       lats <= -75),
    ("antarctic    (-75 < lat <= -60)", (lats > -75) & (lats <= -60)),
    ("S mid-lat    (-60 < lat <= -30)", (lats > -60) & (lats <= -30)),
    ("S tropics    (-30 < lat <= 0)",   (lats > -30) & (lats <= 0)),
    ("N tropics    ( 0 < lat <= 30)",   (lats >  0)  & (lats <= 30)),
    ("N mid-lat    (30 < lat <= 60)",   (lats > 30) & (lats <= 60)),
    ("arctic       (60 < lat <= 75)",   (lats > 60) & (lats <= 75)),
    ("north polar  (lat > 75)",          lats > 75),
]
print(f"\nobserved clusters by latitude band (total: {len(obs)}):")
for name, mask in bands:
    n = int(mask.sum())
    pct = 100 * n / max(len(obs), 1)
    print(f"  {name:<32}  {n:>7} ({pct:>5.1f}%)")

import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lats.values, bins=np.linspace(-90, 90, 37))
ax.axvspan(-23.5, 23.5, alpha=0.15, color="green", label="tropics (|lat|<23.5°)")
ax.set_xlabel("centroid latitude (deg)")
ax.set_ylabel("count of observed clusters")
ax.set_title("Latitude distribution of observed clusters")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# export aggregated_observed -> GeoJSON for downstream viz (geojson.io / kepler.gl / QGIS).
# notes:
#   - GeoJSON (RFC 7946) is WGS84-only, so we explicitly to_crs("EPSG:4326") even
#     though aggregated_observed already inherits ground_tracks.crs.
#   - GeoJSON property values must be JSON-native scalars, so the `time` column
#     (datetime) is serialized to ISO-8601 Z-form strings before write.
#   - solar_hour: if upstream made it a Timedelta, coerce to float hours; otherwise
#     leave it alone. Avoids opaque "ns-int" rendering in GeoJSON property panes.
export_gdf = aggregated_observed.copy()

if pd.api.types.is_datetime64_any_dtype(export_gdf["time"]):
    export_gdf["time"] = export_gdf["time"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

if pd.api.types.is_timedelta64_dtype(export_gdf["solar_hour"]):
    export_gdf["solar_hour"] = export_gdf["solar_hour"].dt.total_seconds() / 3600.0

if export_gdf.crs is None or export_gdf.crs.to_epsg() != 4326:
    export_gdf = export_gdf.to_crs("EPSG:4326")

export_path = "observedclusters_2yr.geojson"
export_gdf.to_file(export_path, driver="GeoJSON")
print(f"wrote {len(export_gdf)} features to {export_path}")


In [ ]:
print(len(aggregated_observed[aggregated_observed["avg_prectot"]>0]))
print(len(joined_observed[joined_observed["avg_prectot"]>0]))
print(len(clusters[clusters["avg_prectot"]>0]))